In [1]:
# ============================================================
# FIX FAISS / OPENMP CRASH
# ============================================================

import os

os.environ["OMP_NUM_THREADS"] = "1"

os.environ["KMP_DUPLICATE_LIB_OK"] = "TRUE"

print(
    "OpenMP Fix Applied"
)

OpenMP Fix Applied


In [2]:
# ============================================================
# IMPORT REQUIRED LIBRARIES
# ============================================================

import pandas as pd

from dotenv import load_dotenv

from langchain_openai import OpenAIEmbeddings

from langchain_openai import ChatOpenAI

from langchain_community.vectorstores import FAISS

print(
    "Libraries Loaded Successfully"
)

Libraries Loaded Successfully


/var/folders/vq/9swrpcvn7sl3cb_pcdyf9r3h0000gn/T/ipykernel_1663/649247855.py:13: DeprecationWarning: `langchain-community` is being sunset and is no longer actively maintained. See https://github.com/langchain-ai/langchain-community/issues/674 for details and migration guidance toward standalone integration packages.
  from langchain_community.vectorstores import FAISS


In [3]:
# ============================================================
# LOAD ENVIRONMENT VARIABLES
# ============================================================

load_dotenv()

print(
    "Environment Variables Loaded"
)

Environment Variables Loaded


In [4]:
# ============================================================
# LOAD EVALUATION DATASET
# ============================================================

eval_df = pd.read_csv(
    "../notebooks/evaluation_dataset.csv"
)

print(
    "Rows:",
    len(eval_df)
)

eval_df.head()

Rows: 1000


,query,article,reference_summary
0,solar observatory aims to provide better under...,cnn nasa has postponed for one day the schedul...,solar observatory aims to provide better under...
1,mary j bliges new album my life ii,cnn seventeen years after the release of her b...,mary j bliges new album my life ii is a follow...
2,germany coach joachim low claims spain are fav...,cnn germany coach joachim low has tried to lif...,germany coach joachim low claims spain are fav...
3,presidential historians weigh in on how histor...,cnn with record low approval ratings and inten...,presidential historians weigh in on how histor...
4,chinas gold medal gymnasts cleared of competin...,cnn chinas olympic gold medal gymnasts have be...,chinas gold medal gymnasts cleared of competin...


In [5]:
# ============================================================
# VERIFY DATASET
# ============================================================

print(
    eval_df.columns.tolist()
)

['query', 'article', 'reference_summary']


In [6]:
# ============================================================
# LOAD OPENAI EMBEDDINGS
# ============================================================

embeddings = OpenAIEmbeddings(

    model="text-embedding-3-small"

)

print(
    "Embeddings Loaded Successfully"
)

Embeddings Loaded Successfully


In [7]:
# ============================================================
# LOAD FAISS VECTOR STORE
# ============================================================

vector_store = FAISS.load_local(

    "../notebooks/faiss_20k",

    embeddings,

    allow_dangerous_deserialization=True

)

print(
    "FAISS Loaded Successfully"
)

FAISS Loaded Successfully


In [9]:
# ============================================================
# TEST RETRIEVAL
# ============================================================

docs = vector_store.similarity_search(

    "artificial intelligence",

    k=5

)

print(
    "Retrieved Documents:",
    len(docs)
)

Retrieved Documents: 5


In [10]:
# ============================================================
# LOAD GPT4o
# ============================================================

llm = ChatOpenAI(

    model="gpt-4o",

    temperature=0

)

print(
    "GPT4o Loaded Successfully"
)

GPT4o Loaded Successfully


In [11]:
# ============================================================
# TEST GPT4o
# ============================================================

response = llm.invoke(
    "Say Hello"
)

print(
    response.content
)

Hello! How can I assist you today?


In [12]:
# ============================================================
# RESET RESULTS
# ============================================================

results = []

print(
    "Results Reset Successfully"
)

Results Reset Successfully


In [13]:
# ============================================================
# DENSE GPT4o EVALUATION PIPELINE
# ============================================================

for idx, row in eval_df.iterrows():

    if idx >= 500:
        break

    if idx % 50 == 0:

        print(
            f"Processing Row {idx}"
        )

    query = str(
        row["query"]
    )

    reference_summary = str(
        row["reference_summary"]
    )

    # --------------------------------------------------------
    # DENSE RETRIEVAL
    # --------------------------------------------------------

    docs = vector_store.similarity_search(

        query,

        k=5

    )

    context = "\n\n".join(

        [

            doc.page_content

            for doc in docs

        ]

    )

    # --------------------------------------------------------
    # PROMPT
    # --------------------------------------------------------

    prompt = f"""

    You are a helpful AI assistant.

    Use ONLY the provided context.

    Context:

    {context}

    Question:

    {query}

    Answer:

    """

    # --------------------------------------------------------
    # GENERATE RESPONSE
    # --------------------------------------------------------

    response = llm.invoke(
        prompt
    )

    answer = response.content

    # --------------------------------------------------------
    # STORE RESULT
    # --------------------------------------------------------

    results.append({

        "pipeline":
        "Dense GPT4o",

        "model":
        "GPT4o",

        "retrieval_method":
        "Dense",

        "query":
        query,

        "reference_summary":
        reference_summary,

        "generated_answer":
        answer

    })

print(
    "\nDense GPT4o Evaluation Completed"
)

Processing Row 0
Processing Row 50
Processing Row 100
Processing Row 150
Processing Row 200
Processing Row 250
Processing Row 300
Processing Row 350
Processing Row 400
Processing Row 450

Dense GPT4o Evaluation Completed


In [14]:
# ============================================================
# CREATE RESULTS DATAFRAME
# ============================================================

results_df = pd.DataFrame(
    results
)

print(
    "Total Records:",
    len(results_df)
)

results_df.head()

Total Records: 500


,pipeline,model,retrieval_method,query,reference_summary,generated_answer
0,Dense GPT4o,GPT4o,Dense,solar observatory aims to provide better under...,solar observatory aims to provide better under...,the sun and its role in space weather events s...
1,Dense GPT4o,GPT4o,Dense,mary j bliges new album my life ii,mary j bliges new album my life ii is a follow...,"Mary J. Blige's new album ""My Life II: The Jou..."
2,Dense GPT4o,GPT4o,Dense,germany coach joachim low claims spain are fav...,germany coach joachim low claims spain are fav...,Germany coach Joachim Low claims that Spain ar...
3,Dense GPT4o,GPT4o,Dense,presidential historians weigh in on how histor...,presidential historians weigh in on how histor...,presidential historians weigh in on how histor...
4,Dense GPT4o,GPT4o,Dense,chinas gold medal gymnasts cleared of competin...,chinas gold medal gymnasts cleared of competin...,China's gold medal gymnasts were cleared of co...


In [15]:
# ============================================================
# VERIFY DATAFRAME STRUCTURE
# ============================================================

results_df.info()

<class 'pandas.DataFrame'>
RangeIndex: 500 entries, 0 to 499
Data columns (total 6 columns):
 #   Column             Non-Null Count  Dtype
---  ------             --------------  -----
 0   pipeline           500 non-null    str  
 1   model              500 non-null    str  
 2   retrieval_method   500 non-null    str  
 3   query              500 non-null    str  
 4   reference_summary  500 non-null    str  
 5   generated_answer   500 non-null    str  
dtypes: str(6)
memory usage: 283.2 KB


In [16]:
# ============================================================
# CHECK NULL VALUES
# ============================================================

results_df.isnull().sum()

pipeline             0
model                0
retrieval_method     0
query                0
reference_summary    0
generated_answer     0
dtype: int64

In [17]:
# ============================================================
# MANUAL VALIDATION
# ============================================================

results_df[

    [

        "query",

        "reference_summary",

        "generated_answer"

    ]

].head(10)

,query,reference_summary,generated_answer
0,solar observatory aims to provide better under...,solar observatory aims to provide better under...,the sun and its role in space weather events s...
1,mary j bliges new album my life ii,mary j bliges new album my life ii is a follow...,"Mary J. Blige's new album ""My Life II: The Jou..."
2,germany coach joachim low claims spain are fav...,germany coach joachim low claims spain are fav...,Germany coach Joachim Low claims that Spain ar...
3,presidential historians weigh in on how histor...,presidential historians weigh in on how histor...,presidential historians weigh in on how histor...
4,chinas gold medal gymnasts cleared of competin...,chinas gold medal gymnasts cleared of competin...,China's gold medal gymnasts were cleared of co...
5,critics and viewers see stewart as victor after,critics and viewers see stewart as victor afte...,critics and viewers see Stewart as the victor ...
6,bahr idriss abu garda faces charges in deaths,bahr idriss abu garda faces charges in deaths ...,Bahr Idriss Abu Garda faces charges related to...
7,south africa beat cohosts india in world cup,south africa beat cohosts india in world cup c...,South Africa beat cohosts India in the World C...
8,rudy ruiz its become unfashionable to have an,rudy ruiz its become unfashionable to have an ...,Rudy Ruiz suggests that it's become unfashiona...
9,fabio cannavaro is to join the italian national,fabio cannavaro is to join the italian nationa...,Fabio Cannavaro is to join the Italian nationa...


In [18]:
# ============================================================
# SAVE EVALUATION RESULTS
# ============================================================

results_df.to_csv(

    "dense_gpt4o_eval.csv",

    index=False

)

print(
    "Dense GPT4o Evaluation Results Saved Successfully"
)

Dense GPT4o Evaluation Results Saved Successfully


In [19]:
# ============================================================
# VERIFY SAVED FILE
# ============================================================

saved_df = pd.read_csv(
    "dense_gpt4o_eval.csv"
)

print(
    saved_df.shape
)

saved_df.head()

(500, 6)


,pipeline,model,retrieval_method,query,reference_summary,generated_answer
0,Dense GPT4o,GPT4o,Dense,solar observatory aims to provide better under...,solar observatory aims to provide better under...,the sun and its role in space weather events s...
1,Dense GPT4o,GPT4o,Dense,mary j bliges new album my life ii,mary j bliges new album my life ii is a follow...,"Mary J. Blige's new album ""My Life II: The Jou..."
2,Dense GPT4o,GPT4o,Dense,germany coach joachim low claims spain are fav...,germany coach joachim low claims spain are fav...,Germany coach Joachim Low claims that Spain ar...
3,Dense GPT4o,GPT4o,Dense,presidential historians weigh in on how histor...,presidential historians weigh in on how histor...,presidential historians weigh in on how histor...
4,Dense GPT4o,GPT4o,Dense,chinas gold medal gymnasts cleared of competin...,chinas gold medal gymnasts cleared of competin...,China's gold medal gymnasts were cleared of co...


Those notebooks were never designed for
ground-truth evaluation.

They were designed for
retrieval and performance testing.